In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

from tsfm_lens.chronos.circuitlens import CircuitLensChronos


## Setup and visualization

In [ ]:
def sinusoidal(
    k: float | list[float],
    t_1: float = 1,
    steps: int = 100,
    amp: float | list[float] = 1,
    phase_shift: list[float] = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Return t, y for sum of sinusoids with frequency k and amplitude amp."""
    t = np.linspace(0, t_1, steps, endpoint=False)
    k_arr = np.atleast_1d(k)
    amp_arr = np.full_like(k_arr, amp, dtype=float) if np.isscalar(amp) and len(k_arr) > 1 else np.atleast_1d(amp)
    if len(k_arr) != len(amp_arr):
        raise ValueError("k and amp must have the same length when passed as arrays")
    phase_shift_arr = (
        np.full_like(k_arr, phase_shift, dtype=float)
        if np.isscalar(phase_shift) and len(k_arr) > 1
        else np.atleast_1d(phase_shift)
    )
    y = sum(
        a * np.sin(2 * np.pi * k_ * t + phase_shift_) for k_, a, phase_shift_ in zip(k_arr, amp_arr, phase_shift_arr)
    )
    return t, y


def geometric_frequencies(init_freqs: list[float], r: float, n: int) -> np.ndarray:
    """
    Create a list of n frequencies spaced geometrically, with the ratio r.
    """
    freqs = np.zeros(n)
    width = init_freqs[1] - init_freqs[0]
    freqs[0] = init_freqs[0]

    for i in range(1, n):
        freqs[i] = width / r ** (i - 1) + init_freqs[0]

    return freqs

In [ ]:
# plot sine geometric as well as the fourier transform of the sine geometric
n = 5

amps = np.array([np.exp(-i / 3) for i in range(n)])
k = geometric_frequencies(init_freqs=(0.1, 0.5), r=0.5, n=n)
# Generate random phase shifts for each frequency component
phase_shifts = np.random.uniform(0, 2 * np.pi, n)
t, y = sinusoidal(k=k, amp=amps, phase_shift=phase_shifts, t_1=20, steps=1024)

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Plot the original signal
ax1.plot(t, y)
ax1.set_title("Sine Geometric Signal")
ax1.set_xlabel("Time")
ax1.set_ylabel("Amplitude")

# Compute the Fourier transform
yf = np.fft.fft(y)
xf = np.fft.fftfreq(len(y), t[1] - t[0])  # Frequency bins
# Only plot the positive frequencies (first half)
n = len(y) // 2
ax2.plot(xf[:n], 2.0 / len(y) * np.abs(yf[:n]))
ax2.set_title("Fourier Transform")
ax2.set_xlabel("Frequency")
ax2.set_ylabel("Magnitude")

plt.tight_layout()
plt.show()

## Main Demo

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map="auto")

In [ ]:
def geometric_multisine_context(
    r: float = 0.5, n: int = 10, init_freqs: list[float] = (0.1, 0.2), seed: int = 42
) -> torch.Tensor:
    np.random.seed(seed)

    amps = np.array([np.exp(-i / 3) for i in range(n)])
    k = geometric_frequencies(init_freqs=init_freqs, r=r, n=n)
    # Generate random phase shifts for each frequency component
    phase_shifts = np.random.uniform(0, 2 * np.pi, n)
    t, y = sinusoidal(k=k, amp=amps, phase_shift=phase_shifts, t_1=20, steps=1024)
    return torch.tensor(t), torch.tensor(y).unsqueeze(0)


t, ground_truth = geometric_multisine_context(r=0.5, n=10, init_freqs=(0.1, 0.2))
context = ground_truth[..., :512]
future = ground_truth[..., 512:]

outputs = pipeline.predict(
    context=context,
    prediction_length=512,
    num_samples=20,
)
pred = outputs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Create a figure with appropriate size
plt.figure(figsize=(12, 6))

# Time steps for plotting
t_context = np.arange(context.shape[-1])
t_future = np.arange(context.shape[-1], context.shape[-1] + future.shape[-1])
t_all = np.arange(context.shape[-1] + future.shape[-1])

# Plot ground truth (context)
plt.plot(t_context, context[0].numpy(), "b-", linewidth=2, label="Ground Truth (Context)")

# Plot ground truth future
plt.plot(t_future, future[0].numpy(), "g-", linewidth=2, label="Ground Truth (Future)")

# Plot each of the 5 prediction samples
for i in range(pred.shape[1]):
    plt.plot(t_future, pred[0, i].numpy(), "r-", alpha=0.3, linewidth=1)

# Calculate and plot the median prediction
median_pred = torch.median(pred, dim=1)[0]
plt.plot(t_future, median_pred[0].numpy(), "r-", linewidth=2, label="Prediction (Median)")

# Add a vertical line to separate context and future
plt.axvline(x=context.shape[-1], color="k", linestyle="--", alpha=0.5)

# Add labels and legend
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.title("Time Series Prediction")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
